# Context-aware toxicity: submission and reproducibility

This notebook defaults to **review and inference** using existing evidence and a bundled local checkpoint. It does not silently launch training. All scores refer to target toxicity, not the parent's label.

Read `SUBMISSION_REPORT.md` for results and `SUBMISSION_HANDOFF.md` for exact recovery, training and annotation steps. Full corrected LoRA training is in `LoRA_Fine_Tuning.ipynb`. Historical runs remain preserved.

**Current run guide: RUN_AND_REPORT.md.** The cleaned-data training and ensemble are separate from this historical benchmark review. Full execution is left to you.

In [ ]:
import os, sys, json
from pathlib import Path
start = Path(os.environ.get("TOXICITY_PROJECT_ROOT", Path.cwd())).resolve()
ROOT = next((p for p in (start, *start.parents) if (p / "contextual/config.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter in the project directory or set TOXICITY_PROJECT_ROOT.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from contextual.config import load_config, location, project_path
cfg = load_config(os.environ.get("TOXICITY_CONFIG", ROOT / "configs/contextual_submission.json"))
RUN_TRAINING = os.environ.get("TOXICITY_RUN_TRAINING", "0") == "1"
print("Root:", ROOT)
print("Training enabled:", RUN_TRAINING)
print("New experiment checkpoint:", location(cfg, "checkpoint"))
if RUN_TRAINING and (cfg.get("adapter", {}).get("enabled") or cfg["data"].get("external_sources")):
    raise ValueError("Use LoRA_Fine_Tuning.ipynb for adapter training and Data_Downloads.ipynb for external data.")

## Data preparation (explicit training mode only)

The existing prepared dataset is frozen for comparisons. Curated Telugu challenge examples are excluded from it. Downloading new data does not automatically merge it into this benchmark. To train the frozen-encoder candidate, set `TOXICITY_RUN_TRAINING=1` before launching the notebook; otherwise this cell reads the existing manifest.

In [ ]:
from contextual.data import download, prepare, read_jsonl
if RUN_TRAINING and not location(cfg, "prepared").exists():
    download(cfg)
    manifest = prepare(cfg)
else:
    manifest = json.loads(location(cfg, "manifest").read_text(encoding="utf-8"))
manifest

In [ ]:
from contextual.evaluation import validate_splits, file_hash
examples = read_jsonl(location(cfg, "prepared"))
splits = validate_splits(examples)
expected = cfg["data"].get("expected_prepared_sha256")
assert not expected or file_hash(location(cfg, "prepared")) == expected, "Prepared data changed; use a new experiment version."
print("Splits:", {k: len(v) for k, v in splits.items()})
print("Examples with ancestry:", sum(len(e["texts"]) > 1 for e in examples))

## Optional frozen-encoder experiment

The baseline and neural model share the exact splits. Thresholds are selected on validation. The cache now hashes actual encoder weights; a legacy cache may rebuild once. Full neural training is intentionally not enabled in review mode.

In [ ]:
if RUN_TRAINING:
    from contextual.experiments import train_baseline
    baseline_report = train_baseline(cfg)
    from contextual.training import train
    report = train(cfg)
else:
    print("Training skipped. Existing results are displayed below.")

## Recorded experiments

Historical LoRA v1 used biased validation selection and different sampling. Its results are exploratory. The v2 row appears only after corrected training finishes. Missing results remain missing.

In [ ]:
import pandas as pd
report_files = {
    "TF-IDF rerun": "submission_tfidf.json",
    "Target-only neural (historical)": "target_only_neural_evaluation.json",
    "Context, dropout 0.2 (historical)": "contextual_candidate_evaluation.json",
    "Context, dropout 0 (historical)": "contextual_no_cdropout_evaluation.json",
    "LoRA v1 (historical)": "contextual_lora_evaluation.json",
    "LoRA v2 (corrected)": "contextual_lora_v2_evaluation.json",
}
summary = []
for name, filename in report_files.items():
    path = ROOT / "reports" / filename
    if path.exists():
        r = json.loads(path.read_text(encoding="utf-8"))
        context = r.get("contextual_test", r.get("contextual_test_subset")) or {}
        summary.append({"model": name, "test_n": r["test"]["n"], "test_F1": r["test"]["f1"],
                        "context_n": context.get("n"), "context_F1": context.get("f1")})
    else:
        summary.append({"model": name, "status": "Not run"})
pd.DataFrame(summary)

## Inference and individual/conversation embeddings

Review mode uses `configs/demo.json` if the new experiment has not been trained. This is an explicitly identified historical checkpoint, not a claim that v2 training completed. No conversation-domain selection is required.

In [ ]:
import torch, numpy as np
from contextual.inference import Predictor
torch.set_num_threads(cfg["training"]["cpu_threads"])
clean_demo = load_config(ROOT / "configs/demo_clean_lora.json")
inference_cfg = (cfg if (location(cfg, "checkpoint") / "metadata.json").exists() else
                 clean_demo if (location(clean_demo, "checkpoint") / "metadata.json").exists() else
                 load_config(ROOT / "configs/demo.json"))
predictor = Predictor(location(inference_cfg, "checkpoint"), device=inference_cfg["training"]["device"])
print("Actual inference checkpoint:", location(inference_cfg, "checkpoint"))
print("Device:", predictor.encoder.device)
sample = next((e for e in splits["test"] if len(e["texts"]) > 1), splits["test"][0])
result = predictor.predict(sample["texts"][-1], sample["texts"][:-1], include_embeddings=True)
print("Individual:", np.asarray(result["individual_embeddings"]).shape)
print("Contextual:", np.asarray(result["contextual_embeddings"]).shape)
print("Conversation:", np.asarray(result["conversation_embedding"]).shape)
{k: v for k, v in result.items() if "embedding" not in k}

In [ ]:
messages = [dict(id=str(i), parent_id=str(i-1) if i else None,
                 conversation_id=sample["group"], text=text) for i, text in enumerate(sample["texts"])]
thread_result = predictor.predict_thread(messages, target_id=messages[-1]["id"])
thread_result

## Current next steps

Follow RUN_AND_REPORT.md for cleaned preprocessing, baselines, LoRA training, ensemble selection, uncertainty analysis and the files to send back. Historical results above do not establish cleaned-data improvement. The GPU recovered, but full training was stopped at your request. The 51 authored challenge targets remain a development diagnostic.